# MVTec Anomaly Detection — All Categories
**Pipeline:** Preprocessing → Harris → Pyramid → SIFT → Segmentation → Classification

Select a category from the Gradio interface and everything runs automatically.

**Dataset:** `/kaggle/input/mvtec-ad/`

## 0. Setup & Imports

In [1]:
import os
import glob
import numpy as np

# ── Matplotlib backend لازم يتعمل قبل أي import تاني ─────────────────────
import matplotlib
matplotlib.use('Agg')          # بدون GUI — مهم جداً في Kaggle + Gradio
import matplotlib.pyplot as plt

import cv2
import io
from PIL import Image
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import copy
import warnings
warnings.filterwarnings('ignore')

# ── Paths ─────────────────────────────────────────────────────────────────
# عدّلي المسار لو الداتاست في مكان تاني
DATASET_ROOT = '/kaggle/input/datasets/ipythonx/mvtec-ad'

ALL_CATEGORIES = [
    'bottle', 'cable', 'capsule', 'carpet', 'grid',
    'hazelnut', 'leather', 'metal_nut', 'pill',
    'screw', 'tile', 'toothbrush', 'transistor',
    'wood', 'zipper'
]

IMG_SIZE = (256, 256)
SEED     = 42
np.random.seed(SEED)

sift = cv2.SIFT_create(nfeatures=300)

print('✓ Imports done')
print(f'Dataset root : {DATASET_ROOT}')
print(f'Exists       : {os.path.exists(DATASET_ROOT)}')

✓ Imports done
Dataset root : /kaggle/input/datasets/ipythonx/mvtec-ad
Exists       : True


## 1. Core Pipeline Functions

In [2]:
# ─────────────────────────────────────────────────────────────────────────
# DATA LOADING
# ─────────────────────────────────────────────────────────────────────────
def load_images(folder, label):
    paths = sorted(glob.glob(os.path.join(folder, '*.png')))
    imgs, labels = [], []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, IMG_SIZE)
        imgs.append(img)
        labels.append(label)
    return imgs, labels


def load_category(category):
    base       = os.path.join(DATASET_ROOT, category)
    train_good = os.path.join(base, 'train', 'good')
    test_dir   = os.path.join(base, 'test')

    if not os.path.isdir(train_good):
        raise FileNotFoundError(f'مسار التدريب مش موجود: {train_good}')

    train_imgs, train_labels = load_images(train_good, label=0)

    test_imgs, test_labels, defect_types = [], [], []
    if os.path.isdir(test_dir):
        defect_types = sorted([
            d for d in os.listdir(test_dir)
            if os.path.isdir(os.path.join(test_dir, d))
        ])
        for dt in defect_types:
            lbl  = 0 if dt == 'good' else 1
            imgs, lbls = load_images(os.path.join(test_dir, dt), label=lbl)
            test_imgs   += imgs
            test_labels += lbls

    return train_imgs, train_labels, test_imgs, test_labels, defect_types


# ─────────────────────────────────────────────────────────────────────────
# PREPROCESSING HELPERS
# ─────────────────────────────────────────────────────────────────────────
def harris_corners(img_rgb, threshold=0.01):
    gray     = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
    response = cv2.cornerHarris(gray, blockSize=2, ksize=3, k=0.04)
    response = cv2.dilate(response, None)
    marked   = img_rgb.copy()
    marked[response > threshold * response.max()] = [255, 0, 0]
    count    = int(np.sum(response > threshold * response.max()))
    return marked, count


def kmeans_segment(img_rgb, k=3):
    # ملحوظة: K-Means بتشتغل على الصورة الملونة RGB
    # مش الـ grayscale — عشان كده بنقدر نستخرج color-based clusters
    pixels   = img_rgb.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.2)
    _, labels, centers = cv2.kmeans(
        pixels, k, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS
    )
    centers  = centers.astype(np.uint8)
    seg_img  = centers[labels.flatten()].reshape(img_rgb.shape)
    lbl_mask = labels.reshape(img_rgb.shape[:2])
    return seg_img, lbl_mask


# ─────────────────────────────────────────────────────────────────────────
# FEATURE EXTRACTION — 22 features per image
#
# ملحوظة عن الـ SIFT:
#   SIFT بيطلع descriptor (128,) لكل keypoint.
#   لو عندنا 200 keypoint → matrix (200, 128).
#   مش ممكن نحط الـ 25600 رقم ده كـ features مباشرة لأن:
#     1) الحجم متغير (كل صورة عندها عدد keypoints مختلف)
#     2) هيبقى أكبر بكتير من الداتا اللي عندنا
#   الحل: بناخد mean + std على كل الـ matrix → رقمين بس
#   ده اسمه "global descriptor statistics" وشائع جداً في CV التقليدي.
#
# ملحوظة عن الـ Color Features:
#   Harris، SIFT، Laplacian → بيشتغلوا على grayscale
#   لكن Color Stats (r_mean, g_mean, ...) → بتاخد من img_rgb الأصلية
#   الصورة الملونة موجودة، بس بنحولها grayscale بس لما نحتاجها
# ─────────────────────────────────────────────────────────────────────────
def extract_features(img_rgb):
    # grayscale للعمليات اللي محتاجاها
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)

    # 1. Filter difference stats (4 features)
    gauss  = cv2.GaussianBlur(gray, (5, 5), 1.5)
    median = cv2.medianBlur(gray.astype(np.uint8), 5).astype(np.float32)
    diff_g = np.abs(gray - gauss)
    diff_m = np.abs(gray - median)

    # 2. Harris corner density (1 feature)
    response       = cv2.cornerHarris(gray, 2, 3, 0.04)
    corner_density = float(np.sum(response > 0.01 * response.max())) / (IMG_SIZE[0] * IMG_SIZE[1])

    # 3. SIFT stats (5 features)
    #    - n_kps           : عدد الـ keypoints
    #    - resp_mean/std   : إحصائيات قوة الـ keypoints
    #    - desc_mean/std   : إحصائيات على مصفوفة الـ descriptors (N×128)
    kps, descs = sift.detectAndCompute(gray.astype(np.uint8), None)
    n_kps      = len(kps)
    resp_mean  = float(np.mean([k.response for k in kps])) if kps else 0.0
    resp_std   = float(np.std ([k.response for k in kps])) if kps else 0.0
    desc_mean  = float(descs.mean()) if descs is not None else 0.0
    desc_std   = float(descs.std())  if descs is not None else 0.0

    # 4. Laplacian edge energy (3 features)
    lap      = cv2.Laplacian(gray.astype(np.uint8), cv2.CV_64F)
    lap_mean = float(np.mean(np.abs(lap)))
    lap_std  = float(np.std(lap))
    lap_var  = float(lap.var())

    # 5. Color stats من الصورة الملونة الأصلية (6 features)
    #    Harris/SIFT/LAP بيشتغلوا على grayscale
    #    لكن img_rgb لسه موجودة وبنستخرج منها الـ color info
    r_mean, g_mean, b_mean = [float(img_rgb[:, :, c].mean()) for c in range(3)]
    r_std,  g_std,  b_std  = [float(img_rgb[:, :, c].std())  for c in range(3)]

    # 6. K-Means cluster ratios من الصورة الملونة (3 features)
    #    K-Means على RGB → بيقسم الصورة حسب الألوان
    _, lbl     = kmeans_segment(img_rgb, k=4)
    counts     = [np.sum(lbl == i) for i in range(3)]
    seg_ratios = [c / lbl.size for c in sorted(counts)]

    # Total: 4 + 1 + 5 + 3 + 6 + 3 = 22 features
    return np.array([
        diff_g.mean(), diff_g.std(), diff_m.mean(), diff_m.std(),   # 4
        corner_density,                                              # 1
        n_kps, resp_mean, resp_std, desc_mean, desc_std,            # 5
        lap_mean, lap_std, lap_var,                                  # 3
        r_mean, g_mean, b_mean, r_std, g_std, b_std,                # 6
        *seg_ratios                                                  # 3
    ], dtype=np.float32)


# ─────────────────────────────────────────────────────────────────────────
# TRAIN PIPELINE
# ─────────────────────────────────────────────────────────────────────────
def train_pipeline(category):
    print(f'=== Loading category: {category} ===')
    train_imgs, train_labels, test_imgs, test_labels, defect_types = load_category(category)
    print(f'  Train: {len(train_imgs)}  |  Test: {len(test_imgs)}')

    if len(train_imgs) == 0:
        raise ValueError(f'مفيش صور في train/good لـ {category}')

    all_imgs   = train_imgs + test_imgs
    all_labels = train_labels + test_labels

    print('  Extracting features...')
    X = np.array([extract_features(img) for img in all_imgs])
    y = np.array(all_labels)
    print(f'  Feature matrix: {X.shape}')

    # تأكد إن عندنا كلتين الـ classes قبل stratify
    stratify_y = y if len(np.unique(y)) > 1 else None

    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=SEED, stratify=stratify_y
    )

    # ── Bug Fix 1: scaler بيتعمل fit على X_train بس ─────────────────────
    scaler   = StandardScaler()
    X_tr_s   = scaler.fit_transform(X_tr)
    X_te_s   = scaler.transform(X_te)

    classifiers = {
        'Naive Bayes'      : GaussianNB(),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
        'Random Forest'    : RandomForestClassifier(n_estimators=100, random_state=SEED),
    }

    results = {}
    trained_clfs = {}
    for name, clf in classifiers.items():
        clf.fit(X_tr_s, y_tr)
        y_pred = clf.predict(X_te_s)
        results[name] = {
            'acc' : accuracy_score (y_te, y_pred),
            'prec': precision_score(y_te, y_pred, zero_division=0),
            'rec' : recall_score   (y_te, y_pred, zero_division=0),
            'f1'  : f1_score       (y_te, y_pred, zero_division=0),
            'cm'  : confusion_matrix(y_te, y_pred),
        }
        trained_clfs[name] = clf   # ── Bug Fix 2: نفصل الـ clf عن results ──
        print(f'  [{name}]  Acc={results[name]["acc"]:.3f}  F1={results[name]["f1"]:.3f}')

    best_name = max(results, key=lambda k: results[k]['f1'])
    print(f'  Best: {best_name}  (F1={results[best_name]["f1"]:.3f})')

    # ── Bug Fix 3: إعادة التدريب على كل الداتا بشكل صح ──────────────────
    # بناخد نسخة جديدة من الموديل بدل ما نعيد استخدام الـ fitted واحد
    # والـ scaler بيتعمل fit على كل X مرة واحدة بس هنا
    best_clf = copy.deepcopy(trained_clfs[best_name].__class__(
        **trained_clfs[best_name].get_params()
    ))
    X_all_s = scaler.fit_transform(X)   # الـ scaler بيتحدث على كل الداتا
    best_clf.fit(X_all_s, y)
    print('  Model re-trained on full dataset ✓')

    return best_clf, scaler, results, best_name, defect_types


print('✓ Core functions loaded')

✓ Core functions loaded


## 2. Gradio Interface

In [3]:
!pip install gradio -q

In [4]:
import gradio as gr
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import io
from PIL import Image

# ── Global state ──────────────────────────────────────────────────────────
_state = {
    'category'  : None,
    'clf'       : None,
    'scaler'    : None,
    'results'   : None,
    'best_name' : None,
    'defects'   : [],
}


def fig_to_pil(fig):
    """Matplotlib figure → PIL Image for Gradio."""
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close(fig)
    return img


# ─────────────────────────────────────────────────────────────────────────
# STEP 1 — Train on selected category
# ─────────────────────────────────────────────────────────────────────────
def train_on_category(category):
    """Returns (status_text, metrics_img, cm_img)."""
    if not category:
        return 'Please select a category first!', None, None

    try:
        clf, scaler, results, best_name, defects = train_pipeline(category)
    except Exception as e:
        return f'Error during training:\n{e}', None, None

    _state.update({
        'category'  : category,
        'clf'       : clf,
        'scaler'    : scaler,
        'results'   : results,
        'best_name' : best_name,
        'defects'   : defects,
    })

    # ── Metrics bar chart ──────────────────────────────────────────────────
    fig1, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig1.suptitle(f'Classifier Performance — {category}', fontsize=13, fontweight='bold')
    colors  = ['#4C72B0', '#DD8452', '#55A868']
    metrics = ['acc', 'prec', 'rec', 'f1']
    mlabels = ['Accuracy', 'Precision', 'Recall', 'F1']
    for ax, (cname, color) in zip(axes, zip(results.keys(), colors)):
        r    = results[cname]
        vals = [r[m] for m in metrics]
        bars = ax.bar(mlabels, vals, color=color, alpha=0.85, edgecolor='white')
        ax.set_ylim(0, 1.12)
        ax.set_title(cname, fontsize=9, fontweight='bold')
        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width() / 2, val + 0.03,
                    f'{val:.2f}', ha='center', va='bottom', fontsize=9)
    plt.tight_layout()
    metrics_img = fig_to_pil(fig1)

    # ── Confusion matrices ────────────────────────────────────────────────
    fig2, axes2 = plt.subplots(1, 3, figsize=(12, 4))
    fig2.suptitle('Confusion Matrices', fontsize=13, fontweight='bold')
    for ax, (cname, color) in zip(axes2, zip(results.keys(), colors)):
        cm = results[cname]['cm']
        ax.imshow(cm, cmap='Blues')
        ax.set_title(cname, fontsize=9)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['Normal', 'Defect'])
        ax.set_yticklabels(['Normal', 'Defect'])
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        for i in range(cm.shape[0]):
            for j in range(cm.shape[1]):
                ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                        color='white' if cm[i, j] > cm.max() / 2 else 'black',
                        fontsize=14)
    plt.tight_layout()
    cm_img = fig_to_pil(fig2)

    # ── Status text ───────────────────────────────────────────────────────
    r = results[best_name]
    status = (
        f'✓ Model trained on: {category}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'Defect types  : {defects}\n'
        f'\nBest model    : {best_name}\n'
        f'  Accuracy    : {r["acc"]:.4f}\n'
        f'  Precision   : {r["prec"]:.4f}\n'
        f'  Recall      : {r["rec"]:.4f}\n'
        f'  F1 Score    : {r["f1"]:.4f}\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'Now upload an image and click Analyze!'
    )
    return status, metrics_img, cm_img


# ─────────────────────────────────────────────────────────────────────────
# STEP 2 — Analyze image (classification + visualizations)
# ─────────────────────────────────────────────────────────────────────────
def analyze_image(pil_image):
    """Returns (result_img, report_text)."""
    if pil_image is None:
        return None, 'Please upload an image first!'
    if _state['clf'] is None:
        return None, 'Please train the model first — select a category and click Train Model!'

    try:
        img_rgb = np.array(pil_image.convert('RGB'))
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)

        feat   = extract_features(img_rgb).reshape(1, -1)
        feat_s = _state['scaler'].transform(feat)
        pred   = _state['clf'].predict(feat_s)[0]
        proba  = (
            _state['clf'].predict_proba(feat_s)[0]
            if hasattr(_state['clf'], 'predict_proba') else None
        )

        label_str  = 'NORMAL ✓'   if pred == 0 else 'DEFECTIVE ✗'
        color_hex  = '#2d8a4e'    if pred == 0 else '#c0392b'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        # Visualizations
        seg, _        = kmeans_segment(img_rgb, k=3)
        harris_m, cnt = harris_corners(img_rgb)
        gray_u8       = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        kps_img, _    = sift.detectAndCompute(gray_u8, None)
        sift_drawn    = cv2.drawKeypoints(
            gray_u8, kps_img, None,
            flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )

        fig, axes = plt.subplots(1, 4, figsize=(18, 4))
        fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(
            f'Category: {_state["category"]}  |  {label_str}  |  Confidence: {confidence}',
            fontsize=13, fontweight='bold', color=color_hex, y=1.03
        )
        axes[0].imshow(img_rgb);                 axes[0].set_title('Input Image');                      axes[0].axis('off')
        axes[1].imshow(seg);                     axes[1].set_title('K-Means Segmentation (K=3)');       axes[1].axis('off')
        axes[2].imshow(harris_m);                axes[2].set_title(f'Harris Corners ({cnt})');          axes[2].axis('off')
        axes[3].imshow(sift_drawn, cmap='gray'); axes[3].set_title(f'SIFT Keypoints ({len(kps_img)})'); axes[3].axis('off')
        plt.tight_layout()
        result_img = fig_to_pil(fig)

        proba_str = ''
        if proba is not None:
            proba_str = (
                f'  P(Normal)    = {proba[0]*100:.1f}%\n'
                f'  P(Defective) = {proba[1]*100:.1f}%\n'
            )

        report = (
            f'============================================\n'
            f'  ANOMALY DETECTION REPORT\n'
            f'============================================\n'
            f'  Category     : {_state["category"]}\n'
            f'  Prediction   : {label_str}\n'
            f'  Confidence   : {confidence}\n'
            f'{proba_str}'
            f'  Harris corners : {cnt}\n'
            f'  SIFT keypoints : {len(kps_img)}\n'
            f'  Best model     : {_state["best_name"]}\n'
            f'============================================'
        )
        return result_img, report

    except Exception as e:
        return None, f'Error during analysis:\n{e}'


# ─────────────────────────────────────────────────────────────────────────
# DEFECT DETECTION — highlight anomalous regions via residual heatmap
# ─────────────────────────────────────────────────────────────────────────
def detect_defects(pil_image):
    """
    High-accuracy defect localisation pipeline.

    Score map is built from FIVE complementary signals:
      1. Multi-scale Gaussian residual   — slow texture deviation at 3 blur radii
      2. Laplacian edge energy           — sharp cracks / scratches
      3. Gabor texture energy            — periodic-texture anomalies (4 orientations)
      4. HSV saturation anomaly          — colour-shifted defect spots
      5. Local standard-deviation map    — high-variance noise / roughness

    Signals are weighted, combined, then refined with:
      - Foreground mask (Otsu) to suppress background/border noise
      - CLAHE-based local contrast normalisation
      - Adaptive (Sauvola-style) thresholding instead of a global percentile
      - Two-pass morphology (close small gaps → open dust speckles)
      - Contour-level confidence scoring (mean score inside each blob)
    """
    if pil_image is None:
        return None, 'Please upload an image first!'
    if _state['clf'] is None:
        return None, 'Please train the model first — select a category and click Train Model!'

    try:
        img_rgb = np.array(pil_image.convert('RGB'))
        img_rgb = cv2.resize(img_rgb, IMG_SIZE)
        gray    = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY).astype(np.float32)
        gray_u8 = gray.astype(np.uint8)

        # ── 1. Classification ─────────────────────────────────────────────
        feat   = extract_features(img_rgb).reshape(1, -1)
        feat_s = _state['scaler'].transform(feat)
        pred   = _state['clf'].predict(feat_s)[0]
        proba  = (
            _state['clf'].predict_proba(feat_s)[0]
            if hasattr(_state['clf'], 'predict_proba') else None
        )
        label_str = 'NORMAL ✓'  if pred == 0 else 'DEFECTIVE ✗'
        color_hex = '#2d8a4e'   if pred == 0 else '#c0392b'
        confidence = f'{max(proba)*100:.1f}%' if proba is not None else 'N/A'

        def norm01(arr):
            mn, mx = arr.min(), arr.max()
            return (arr - mn) / (mx - mn + 1e-8)

        # ── 2. Foreground mask (suppress object border / background) ──────
        # Otsu threshold on the gray image; dilate slightly so we don't
        # accidentally clip real defects that sit right on the boundary.
        _, fg_mask = cv2.threshold(gray_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        # Some categories have dark objects on bright bg — invert if needed
        if np.sum(fg_mask > 0) < fg_mask.size * 0.3:
            fg_mask = cv2.bitwise_not(fg_mask)
        fg_mask = cv2.dilate(fg_mask, np.ones((15, 15), np.uint8), iterations=2)
        fg_float = (fg_mask > 0).astype(np.float32)

        # ── 3. Signal 1: Multi-scale Gaussian residual ────────────────────
        # Three blur radii — coarse, medium, fine — then average residuals
        residuals = []
        for ksize, sigma in [(7, 1.5), (15, 3.0), (31, 6.0)]:
            blurred   = cv2.GaussianBlur(gray, (ksize, ksize), sigma)
            residuals.append(np.abs(gray - blurred))
        sig1 = norm01(np.mean(residuals, axis=0))

        # ── 4. Signal 2: Laplacian edge energy ───────────────────────────
        lap  = np.abs(cv2.Laplacian(gray_u8, cv2.CV_64F, ksize=3)).astype(np.float32)
        # Also compute on a slightly blurred version and take max
        lap2 = np.abs(cv2.Laplacian(
            cv2.GaussianBlur(gray_u8, (5, 5), 1), cv2.CV_64F, ksize=5
        )).astype(np.float32)
        sig2 = norm01(np.maximum(lap, lap2))

        # ── 5. Signal 3: Gabor texture energy (4 orientations) ───────────
        gabor_energy = np.zeros_like(gray)
        for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
            kern = cv2.getGaborKernel(
                (21, 21), sigma=4.0, theta=theta,
                lambd=8.0, gamma=0.5, psi=0, ktype=cv2.CV_32F
            )
            filtered      = cv2.filter2D(gray, cv2.CV_32F, kern)
            gabor_energy  = np.maximum(gabor_energy, np.abs(filtered))
        sig3 = norm01(gabor_energy)

        # ── 6. Signal 4: HSV saturation anomaly ──────────────────────────
        hsv        = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV).astype(np.float32)
        sat        = hsv[:, :, 1]
        sat_blur   = cv2.GaussianBlur(sat, (31, 31), 8.0)
        sig4       = norm01(np.abs(sat - sat_blur))

        # ── 7. Signal 5: Local standard-deviation map ─────────────────────
        # Compute std in a local neighbourhood via (E[X^2] - E[X]^2)^0.5
        ksize_loc  = 15
        blur_sq    = cv2.blur((gray ** 2), (ksize_loc, ksize_loc))
        blur_mean  = cv2.blur(gray,         (ksize_loc, ksize_loc))
        local_var  = np.maximum(blur_sq - blur_mean ** 2, 0)
        sig5       = norm01(np.sqrt(local_var))

        # ── 8. Weighted fusion ────────────────────────────────────────────
        # Weights tuned empirically: Gaussian residual and local std carry
        # the most signal; Gabor helps with periodic-texture categories.
        score_map = (
            0.30 * sig1 +   # multi-scale texture deviation
            0.20 * sig2 +   # edge sharpness
            0.20 * sig3 +   # Gabor texture anomaly
            0.15 * sig4 +   # colour saturation shift
            0.15 * sig5     # local roughness
        )
        score_map = norm01(score_map)

        # Apply foreground mask — zero out background contributions
        score_map = score_map * fg_float

        # CLAHE-style local contrast enhancement on the score map
        score_u8_pre  = (score_map * 255).astype(np.uint8)
        clahe         = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))
        score_u8_enh  = clahe.apply(score_u8_pre)
        score_map     = norm01(score_u8_enh.astype(np.float32))

        # ── 9. Coloured heatmap overlay ───────────────────────────────────
        score_u8 = (score_map * 255).astype(np.uint8)
        heatmap  = cv2.applyColorMap(score_u8, cv2.COLORMAP_INFERNO)   # sharper contrast than JET
        heatmap  = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay  = cv2.addWeighted(img_rgb, 0.50, heatmap, 0.50, 0)

        # ── 10. Region detection — always annotate, boxes on every defect ──
        annotated      = img_rgb.copy()
        defect_regions = []

        # ── helper: draw corner-bracket box (more visible than a plain rect) ──
        def draw_box(img, x, y, w, h, color=(220, 30, 30), thickness=2):
            """Full rectangle + bright corner brackets for maximum visibility."""
            # Full bounding rectangle
            cv2.rectangle(img, (x, y), (x + w, y + h), color, thickness)
            # Corner bracket length = 20% of the shorter side, min 8 px
            cl = max(8, int(min(w, h) * 0.20))
            pts = [(x, y), (x+w, y), (x, y+h), (x+w, y+h)]
            dirs = [(1,1), (-1,1), (1,-1), (-1,-1)]
            for (px, py), (dx, dy) in zip(pts, dirs):
                cv2.line(img, (px, py), (px + dx*cl, py), color, thickness + 2)
                cv2.line(img, (px, py), (px, py + dy*cl), color, thickness + 2)

        if pred == 1:
            # ── Adaptive threshold — compares each pixel to its local mean ──
            binary = cv2.adaptiveThreshold(
                score_u8, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                cv2.THRESH_BINARY,
                blockSize=31, C=-10          # less aggressive C → more blobs kept
            )
            binary = cv2.bitwise_and(binary, fg_mask)

            # Two-pass morphology
            k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
            k_open  = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5,  5))
            binary  = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k_close)
            binary  = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k_open)

            contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            img_area = IMG_SIZE[0] * IMG_SIZE[1]
            min_area = img_area * 0.001   # 0.1% — keep even small defects
            max_area = img_area * 0.80

            # Score every contour by mean anomaly score inside it
            scored = []
            for cnt_c in contours:
                area = cv2.contourArea(cnt_c)
                if area < min_area or area > max_area:
                    continue
                mask_c = np.zeros(score_map.shape, np.uint8)
                cv2.drawContours(mask_c, [cnt_c], -1, 255, -1)
                mean_score_region = float(score_map[mask_c > 0].mean())
                scored.append((mean_score_region, cnt_c))

            scored.sort(key=lambda x: x[0], reverse=True)

            # Fallback: if adaptive threshold finds nothing, use top-10% percentile blobs
            if not scored:
                thresh_val = float(np.percentile(score_map[fg_float > 0], 88))
                binary_fb  = (score_map > thresh_val).astype(np.uint8) * 255
                binary_fb  = cv2.bitwise_and(binary_fb, fg_mask)
                binary_fb  = cv2.morphologyEx(binary_fb, cv2.MORPH_CLOSE, k_close)
                contours_fb, _ = cv2.findContours(binary_fb, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                for cnt_c in contours_fb:
                    area = cv2.contourArea(cnt_c)
                    if area < min_area or area > max_area:
                        continue
                    mask_c = np.zeros(score_map.shape, np.uint8)
                    cv2.drawContours(mask_c, [cnt_c], -1, 255, -1)
                    mean_score_region = float(score_map[mask_c > 0].mean())
                    scored.append((mean_score_region, cnt_c))
                scored.sort(key=lambda x: x[0], reverse=True)

            # Accept regions above 0.20 score, or at least the top 3 regardless
            score_threshold = 0.20
            kept = [item for item in scored if item[0] >= score_threshold]
            if not kept and scored:
                kept = scored[:3]   # guarantee at least 1-3 boxes always appear

            for region_score, cnt_c in kept:
                x, y, w, h = cv2.boundingRect(cnt_c)
                defect_regions.append((x, y, w, h, region_score))

                # Color intensity scales with confidence (bright red → orange)
                intensity = min(1.0, region_score / 0.7)
                r = 220
                g = int(30 + (1 - intensity) * 160)   # orange tint for lower scores
                b = 30
                box_color = (r, g, b)

                draw_box(annotated, x, y, w, h, color=box_color, thickness=2)

                # Label: "DEFECT 0.72" — white text on dark background for readability
                label_txt = f'DEFECT {region_score:.2f}'
                (tw, th), _ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.42, 1)
                lx, ly = x, max(y - 4, th + 2)
                cv2.rectangle(annotated, (lx, ly - th - 2), (lx + tw + 4, ly + 2), box_color, -1)
                cv2.putText(annotated, label_txt, (lx + 2, ly),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)

            # Overall verdict banner at top-left
            banner = f'{label_str}  ({len(kept)} region(s))'
            cv2.rectangle(annotated, (0, 0), (len(banner)*9 + 8, 28), (220, 30, 30), -1)
            cv2.putText(annotated, banner, (4, 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)
        else:
            # Normal image — green banner, no boxes
            banner = label_str
            cv2.rectangle(annotated, (0, 0), (len(banner)*9 + 8, 28), (45, 138, 78), -1)
            cv2.putText(annotated, banner, (4, 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 1, cv2.LINE_AA)

        # ── 11. Side-by-side figure ───────────────────────────────────────
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        fig.patch.set_facecolor('#f8f9fa')
        fig.suptitle(
            f'Defect Detection  |  Category: {_state["category"]}  |  '
            f'{label_str}  |  Confidence: {confidence}',
            fontsize=12, fontweight='bold', color=color_hex
        )
        axes[0].imshow(img_rgb);   axes[0].set_title('Input Image');                    axes[0].axis('off')
        axes[1].imshow(overlay);   axes[1].set_title('Anomaly Heatmap (INFERNO)');       axes[1].axis('off')
        axes[2].imshow(annotated); axes[2].set_title('Defect Regions (scored)');         axes[2].axis('off')

        sm = plt.cm.ScalarMappable(cmap='inferno', norm=plt.Normalize(0, 1))
        sm.set_array([])
        plt.colorbar(sm, ax=axes[1], fraction=0.046, pad=0.04).set_label('Anomaly Score')
        plt.tight_layout()
        result_pil = fig_to_pil(fig)

        # ── 12. Report ────────────────────────────────────────────────────
        proba_str = ''
        if proba is not None:
            proba_str = (
                f'  P(Normal)      = {proba[0]*100:.1f}%\n'
                f'  P(Defective)   = {proba[1]*100:.1f}%\n'
            )

        region_str = ''
        if defect_regions:
            region_str = f'  Defect regions : {len(defect_regions)} area(s) detected\n'
            for idx, (x, y, w, h, rs) in enumerate(defect_regions, 1):
                region_str += f'    Region {idx}: x={x}, y={y}, w={w}, h={h}  score={rs:.3f}\n'
        else:
            region_str = '  Defect regions : None detected\n'

        report = (
            f'============================================\n'
            f'  DEFECT DETECTION REPORT\n'
            f'============================================\n'
            f'  Category       : {_state["category"]}\n'
            f'  Verdict        : {label_str}\n'
            f'  Confidence     : {confidence}\n'
            f'{proba_str}'
            f'  Anomaly score (mean) : {float(score_map.mean()):.4f}\n'
            f'  Anomaly score (max)  : {float(score_map.max()):.4f}\n'
            f'{region_str}'
            f'  Model used     : {_state["best_name"]}\n'
            f'  Signals used   : Gaussian residual (x3 scales) + Laplacian\n'
            f'                   + Gabor (4 angles) + HSV saturation + Local std\n'
            f'============================================'
        )
        return result_pil, report

    except Exception as e:
        import traceback
        return None, f'Error during defect detection:\n{traceback.format_exc()}'


# ─────────────────────────────────────────────────────────────────────────
# GRADIO LAYOUT
# ─────────────────────────────────────────────────────────────────────────
with gr.Blocks(
    title='MVTec AD — Multi-Category Anomaly Detector',
    theme=gr.themes.Soft()
) as demo:

    gr.Markdown("""
    # MVTec Anomaly Detection — All Categories
    **Steps:**
    1. Select a category (Bottle, Cable, Capsule, ...)
    2. Click **Train Model** — the model will train automatically
    3. Go to **Analyze Image** or **Defect Detection** to inspect an image
    """)

    # ── Training section ───────────────────────────────────────────────
    with gr.Row():
        with gr.Column(scale=2):
            cat_dropdown = gr.Dropdown(
                choices=ALL_CATEGORIES,
                label='Select Category',
                value='bottle',
            )
            train_btn = gr.Button('Train Model', variant='primary', size='lg')
        with gr.Column(scale=3):
            train_status = gr.Textbox(
                label='Training Status',
                lines=10,
                placeholder='Select a category and click Train...'
            )

    with gr.Row():
        metrics_out = gr.Image(label='Classifier Metrics')
        cm_out      = gr.Image(label='Confusion Matrices')

    gr.Markdown('---')

    # ── Tabs: Analyze Image  |  Defect Detection ──────────────────────
    with gr.Tabs():

        # ── Tab 1: Full Analysis (original tab) ───────────────────────
        with gr.TabItem('Analyze Image'):
            gr.Markdown('### Upload an image to run the full analysis pipeline')
            with gr.Row():
                with gr.Column(scale=1):
                    inp_img     = gr.Image(type='pil', label='Upload image from test/', height=280)
                    analyze_btn = gr.Button('Analyze Image', variant='secondary', size='lg')
                    gr.Markdown("""
                    **Expected results:**
                    - `test/good`  → NORMAL ✓
                    - Other folders → DEFECTIVE ✗
                    """)
                with gr.Column(scale=2):
                    out_vis  = gr.Image(label='Visual Analysis', height=350)
                    out_text = gr.Textbox(label='Detection Report', lines=12, show_copy_button=True)

        # ── Tab 2: Defect Detection (new) ─────────────────────────────
        with gr.TabItem('Defect Detection'):
            gr.Markdown("""
            ### Defect Detection — Localize anomalous regions
            Upload an image and the model will:
            - Classify it as **Normal** or **Defective**
            - Generate a pixel-level **anomaly heatmap** (Gaussian residual + Laplacian energy)
            - Draw **bounding boxes** around detected defect regions (when Defective)
            """)
            with gr.Row():
                with gr.Column(scale=1):
                    dd_img = gr.Image(type='pil', label='Upload image', height=280)
                    dd_btn = gr.Button('Detect Defects', variant='primary', size='lg')
                    gr.Markdown("""
                    **Heatmap legend:**
                    - 🔵 Blue → low anomaly score (likely normal)
                    - 🔴 Red  → high anomaly score (likely defective)
                    """)
                with gr.Column(scale=2):
                    dd_vis    = gr.Image(label='Defect Detection Result', height=380)
                    dd_report = gr.Textbox(label='Defect Detection Report', lines=14, show_copy_button=True)

    # ── Event wiring ───────────────────────────────────────────────────
    train_btn.click(
        fn=train_on_category,
        inputs=[cat_dropdown],
        outputs=[train_status, metrics_out, cm_out]
    )
    analyze_btn.click(
        fn=analyze_image,
        inputs=[inp_img],
        outputs=[out_vis, out_text]
    )
    dd_btn.click(
        fn=detect_defects,
        inputs=[dd_img],
        outputs=[dd_vis, dd_report]
    )

demo.launch(share=True)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://dce44c01f9af90f2e9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
